# Aprendizado de Máquina — Aula prática E2

## Redução de Dimensionalidade: PCA e t-SNE

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

A Aula E1 agrupou **linhas**. Esta agrupa, em certo sentido, **colunas**: procura as
poucas direções em que os dados de fato variam, e joga fora o resto.

> **A pergunta não é "quais colunas descartar", e sim "em quais direções olhar" —
> e uma direção pode ser uma combinação de todas as colunas.**

É a ferramenta que a Aula 05 pediu quando falamos de **redundância**. Lá medimos que
as 81 colunas do `superconductivity.csv` ocupam pouquíssimas direções; agora vamos
encontrá-las. E vamos terminar com o t-SNE, que serve para uma coisa muito
específica — desenhar — e é rotineiramente usado para outras, com consequências.

### Objetivos

Ao final deste notebook você deve ser capaz de:

- calcular componentes principais **pela SVD, à mão**, e conferir contra o `PCA`;
- interpretar a variância explicada e usar `n_components=0.90`;
- reconstruir dados a partir de poucos componentes e medir o erro;
- explicar por que o PCA **precisa** de padronização;
- usar t-SNE para visualizar, e **medir** por que as distâncias globais nele não
  significam nada;
- pôr o PCA dentro de um `Pipeline` e medir o vazamento de não fazê-lo.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

Os objetos novos são o `PCA` e o `TSNE`. O `pdist` do `scipy` aparece na Seção 6,
para comparar distâncias antes e depois de cada projeção.

In [ ]:
import sklearn.linear_model as skl
import sklearn.model_selection as skm
from scipy.spatial.distance import pdist
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
import warnings
warnings.filterwarnings("ignore")

---
## 2. Os eixos principais, vistos

Comecemos com uma nuvem em duas dimensões, onde dá para ver o que o PCA faz. Ele
procura a direção de **maior variância**, depois a de maior variância entre as
ortogonais a essa, e assim por diante.

In [ ]:
rng = np.random.default_rng(2)
n = 400
Z = rng.normal(size=(n, 2)) * np.array([3.0, 0.7])
ang = np.deg2rad(35)
R = np.array([[np.cos(ang), -np.sin(ang)], [np.sin(ang), np.cos(ang)]])
X2 = Z @ R.T + np.array([4.0, 1.0])

pca2 = PCA().fit(X2)
print("variancia explicada por componente:", np.round(pca2.explained_variance_, 3))
print("proporcao                         :", np.round(pca2.explained_variance_ratio_, 4))
print("direcoes (linhas = componentes):\n", np.round(pca2.components_, 3))

fig, ax = subplots(figsize=(5.0, 4.2))
ax.scatter(X2[:, 0], X2[:, 1], s=8, alpha=0.5)
for i, (vec, var) in enumerate(zip(pca2.components_, pca2.explained_variance_)):
    seta = vec * 2.5 * np.sqrt(var)
    ax.annotate("", xy=pca2.mean_ + seta, xytext=pca2.mean_,
                arrowprops=dict(arrowstyle="->", lw=2.2,
                                color=["crimson", "green"][i]))
    ax.text(*(pca2.mean_ + seta * 1.1), f"CP{i+1}", fontsize=10,
            color=["crimson", "green"][i])
ax.set_aspect("equal"); ax.set_xlabel("x1"); ax.set_ylabel("x2")

As duas setas são ortogonais e têm comprimento proporcional ao desvio-padrão ao
longo de cada direção. O primeiro componente pega quase toda a variação — que é
outro jeito de dizer que estes dados, apesar de terem duas colunas, são
**quase unidimensionais**.

---
## 3. PCA é SVD: fazendo à mão

O PCA não é um algoritmo novo, é uma decomposição de álgebra linear. Centrando os
dados em $\bar{X}$ e escrevendo a decomposição em valores singulares
$X_c = U S V^\top$:

- as **linhas de $V^\top$** são os componentes principais (as direções);
- os **escores** são $X_c V = U S$;
- a variância explicada pelo $j$-ésimo componente é $s_j^2/(n-1)$.

Vamos fazer as duas rotas e comparar.

In [ ]:
Xc = X2 - X2.mean(axis=0)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)

print("direcoes pela SVD:\n", np.round(Vt, 4))
print("direcoes pelo PCA:\n", np.round(pca2.components_, 4))
print(f"\nvariancia pela SVD: {np.round(S ** 2 / (n - 1), 4)}")
print(f"variancia pelo PCA: {np.round(pca2.explained_variance_, 4)}")

escores_mao = Xc @ Vt.T
escores_pca = pca2.transform(X2)
print(f"\nmaior diferenca entre os escores: "
      f"{np.abs(np.abs(escores_mao) - np.abs(escores_pca)).max():.2e}")

São a mesma coisa. O valor absoluto na última comparação existe porque o **sinal**
de um componente é arbitrário: se $v$ é uma direção de máxima variância, $-v$
também é. Nunca interprete o sinal de um componente principal — só a magnitude
relativa das cargas dentro dele.

---
## 4. Quantos componentes? A variância explicada

Agora um conjunto de verdade: os dígitos manuscritos do `scikit-learn`, 1797
imagens de $8\times8$ *pixels*, isto é, **64 colunas**.

In [ ]:
digitos = load_digits()
Xd, yd = digitos.data, digitos.target
print(f"{Xd.shape[0]} imagens de {int(np.sqrt(Xd.shape[1]))}x{int(np.sqrt(Xd.shape[1]))} pixels "
      f"= {Xd.shape[1]} colunas")

pca = PCA().fit(Xd)
acum = np.cumsum(pca.explained_variance_ratio_)

fig, (ax1, ax2) = subplots(1, 2, figsize=(7.8, 3.0))
ax1.plot(np.arange(1, 65), pca.explained_variance_ratio_, "o-", ms=3)
ax1.set_xlabel("componente"); ax1.set_ylabel("proporcao da variancia")
ax1.set_title("scree plot", fontsize=9)
ax2.plot(np.arange(1, 65), acum, "o-", ms=3)
for alvo in [0.80, 0.90, 0.95]:
    k = int(np.searchsorted(acum, alvo)) + 1
    ax2.axhline(alvo, ls=":", lw=0.9, color="gray")
    ax2.annotate(f"{alvo:.0%} -> {k} comp.", xy=(k, alvo), xytext=(6, -12),
                 textcoords="offset points", fontsize=7.5)
ax2.set_xlabel("componentes"); ax2.set_ylabel("variancia acumulada")
ax2.set_title("acumulada", fontsize=9)

for alvo in [0.80, 0.90, 0.95, 0.99]:
    print(f"{alvo:.0%} da variancia: {int(np.searchsorted(acum, alvo)) + 1} componentes de 64")

O `scikit-learn` aceita a proporção diretamente, o que é mais honesto do que chutar
um número de componentes:

In [ ]:
p90 = PCA(n_components=0.90).fit(Xd)
print(f"PCA(n_components=0.90) escolheu {p90.n_components_} componentes")
print(f"variancia efetivamente retida  : {p90.explained_variance_ratio_.sum():.4f}")

---
## 5. Reconstruir: o que se perde

O PCA é uma projeção, e projeções perdem informação. Mas dá para voltar — de forma
aproximada — e olhar o estrago com os próprios olhos.

In [ ]:
fig, axes = subplots(5, 8, figsize=(9, 6))
quais = [0, 1, 2, 3, 4, 5, 6, 7]
for j, q in enumerate(quais):
    axes[0, j].imshow(Xd[q].reshape(8, 8), cmap="gray_r")
    axes[0, j].set_xticks([]); axes[0, j].set_yticks([])
axes[0, 0].set_ylabel("original", fontsize=8)

for linha, k in enumerate([1, 5, 15, 40], start=1):
    p = PCA(n_components=k).fit(Xd)
    recon = p.inverse_transform(p.transform(Xd))
    erro = np.mean((recon - Xd) ** 2)
    for j, q in enumerate(quais):
        axes[linha, j].imshow(recon[q].reshape(8, 8), cmap="gray_r")
        axes[linha, j].set_xticks([]); axes[linha, j].set_yticks([])
    axes[linha, 0].set_ylabel(f"k={k}", fontsize=8)
    print(f"k = {k:2d} componentes: EQM da reconstrucao = {erro:7.3f}   "
          f"variancia retida = {p.explained_variance_ratio_.sum():.3f}")

Com 15 dos 64 componentes os dígitos já são perfeitamente legíveis. Isso é a
redundância da Aula 05 aparecendo: *pixels* vizinhos de uma imagem são quase
iguais, e as 64 colunas descrevem muito menos que 64 direções independentes.

Vale ver também o que são os componentes. Cada um é um vetor de 64 números, ou
seja, **uma imagem** — e as imagens que aparecem são padrões de traço:

In [ ]:
fig, axes = subplots(2, 8, figsize=(9, 2.6))
for j, ax in enumerate(axes.ravel()):
    ax.imshow(pca.components_[j].reshape(8, 8), cmap="RdBu_r")
    ax.set_title(f"CP{j+1}\n{pca.explained_variance_ratio_[j]:.1%}", fontsize=7)
    ax.set_xticks([]); ax.set_yticks([])

---
## 6. t-SNE: bom para desenhar, ruim para o resto

O PCA é **linear**: cada componente é uma combinação linear das colunas. Quando a
estrutura interessante é curva, ele não a captura. O **t-SNE** é não linear e serve
para uma coisa: produzir um desenho em duas dimensões em que pontos parecidos
fiquem perto.

Vamos comparar os dois nos dígitos.

In [ ]:
sub = np.random.default_rng(0).choice(len(Xd), 1200, replace=False)
Xs, ys = Xd[sub], yd[sub]

Z_pca = PCA(n_components=2).fit_transform(Xs)
Z_tsne = TSNE(n_components=2, perplexity=30, init="pca",
              random_state=0).fit_transform(Xs)

fig, (ax1, ax2) = subplots(1, 2, figsize=(10, 4.2))
for ax, Zp, nome in [(ax1, Z_pca, "PCA (2 componentes)"),
                     (ax2, Z_tsne, "t-SNE (perplexity=30)")]:
    sc = ax.scatter(Zp[:, 0], Zp[:, 1], c=ys, cmap="tab10", s=9)
    ax.set_title(nome, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(sc, ax=ax2, ticks=range(10), label="digito")

O t-SNE separa os dez dígitos em ilhas nítidas; o PCA os embaralha, porque duas
direções lineares não bastam para separar dez classes de imagens.

E é exatamente aqui que começam os abusos. O t-SNE é otimizado para preservar
**vizinhanças**, e nada mais. As distâncias entre ilhas, os tamanhos das ilhas e as
posições relativas **não têm interpretação**. Dá para medir isso: comparemos, para
cada projeção, quanto as distâncias no plano se parecem com as distâncias originais.

In [ ]:
from sklearn.neighbors import NearestNeighbors


def vizinhos_preservados(A, B, k=20):
    """Fracao dos k vizinhos de cada ponto que sobrevivem a projecao."""
    va = NearestNeighbors(n_neighbors=k + 1).fit(A).kneighbors(A, return_distance=False)[:, 1:]
    vb = NearestNeighbors(n_neighbors=k + 1).fit(B).kneighbors(B, return_distance=False)[:, 1:]
    return np.mean([len(set(a) & set(b)) / k for a, b in zip(va, vb)])


amostra = np.random.default_rng(1).choice(len(Xs), 400, replace=False)
d_orig = pdist(Xs[amostra])

print(f"{'projecao':8s} {'vizinhos preservados (local)':>30s} {'corr. das distancias (global)':>32s}")
for nome, Zp in [("PCA", Z_pca), ("t-SNE", Z_tsne)]:
    local = vizinhos_preservados(Xs, Zp)
    glob = np.corrcoef(d_orig, pdist(Zp[amostra]))[0, 1]
    print(f"{nome:8s} {local:30.3f} {glob:32.3f}")

fig, (ax1, ax2) = subplots(1, 2, figsize=(7.8, 3.2))
for ax, Zp, nome in [(ax1, Z_pca, "PCA"), (ax2, Z_tsne, "t-SNE")]:
    ax.scatter(d_orig, pdist(Zp[amostra]), s=1, alpha=0.06)
    ax.set_xlabel("distancia no espaco original (64 dim.)")
    ax.set_ylabel(f"distancia na projecao")
    ax.set_title(nome, fontsize=9)

> **A lição.** As duas colunas contam histórias opostas, e é essa oposição que
> define para que serve cada método.
>
> **Localmente o t-SNE ganha de longe**: quase dois terços dos vinte vizinhos mais
> próximos de cada ponto continuam vizinhos depois da projeção, contra menos de um
> quarto no PCA. É exatamente o que ele foi construído para fazer.
>
> **Globalmente o PCA ganha**: a correlação entre as distâncias originais e as
> projetadas é maior, e a figura da esquerda tem uma borda inferior nítida — projeção
> ortogonal só **encurta** distâncias, nunca as alonga. A do t-SNE é um borrão sem
> essa estrutura.
>
> Consequência prática, e é séria: **não meça nada em cima de um t-SNE.** Não rode
> $k$-médias nas coordenadas do t-SNE, não calcule silhueta nelas, não conclua que
> "estes dois grupos são mais parecidos entre si porque estão próximos no gráfico".
> O t-SNE é uma figura, não um espaço.

In [ ]:
fig, axes = subplots(1, 4, figsize=(11, 2.9))
for ax, perp in zip(axes, [2, 10, 30, 100]):
    Zt = TSNE(n_components=2, perplexity=perp, init="pca",
              random_state=0).fit_transform(Xs)
    ax.scatter(Zt[:, 0], Zt[:, 1], c=ys, cmap="tab10", s=6)
    ax.set_title(f"perplexity = {perp}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

A `perplexity` é, grosso modo, quantos vizinhos cada ponto considera. Com valor
pequeno o desenho se fragmenta em grumos que não significam nada; com valor grande
tudo se funde. Não há validação cruzada para escolhê-la — a única saída é olhar
vários valores e desconfiar de qualquer estrutura que só apareça em um deles.

> **Sua vez.** Rode o t-SNE duas vezes com `random_state` diferente e a mesma
> `perplexity`. As figuras são iguais? E os agrupamentos que você enxerga nelas? Faça
> o mesmo com o PCA. A diferença entre os dois comportamentos é o resumo desta seção.

---
## 7. PCA dentro do `Pipeline` — e o vazamento de não fazê-lo

O PCA aprende direções **a partir dos dados**. Pela regra da Aula 07, isso o
qualifica como etapa do modelo, e portanto ele tem de ser reajustado dentro de cada
dobra. A Aula 07 mediu que padronizar fora da dobra é inofensivo; o PCA **não** é o
mesmo caso, porque ele olha todas as colunas ao mesmo tempo — está mais perto da
seleção de variáveis que da padronização.

Vamos medir, no cenário cruel da Aula 07: $y$ é ruído puro.

In [ ]:
rng_v = np.random.default_rng(5)
n_v, d_v, n_rep = 60, 400, 40
dobras = skm.KFold(5, shuffle=True, random_state=0)

r2_errado, r2_certo = [], []
for _ in range(n_rep):
    Xv = rng_v.normal(size=(n_v, d_v))
    yv = rng_v.normal(size=n_v)                     # nenhuma relacao com Xv (R^2 verdadeiro = 0)

    # ERRADO: o PCA ve o conjunto inteiro antes da validacao cruzada
    Zv = PCA(n_components=10).fit_transform(Xv)
    r2_errado.append(skm.cross_val_score(skl.LinearRegression(), Zv, yv,
                                         cv=dobras, scoring="r2").mean())
    # CERTO: o PCA e' uma etapa do pipeline
    pipe = Pipeline([("pca", PCA(n_components=10)), ("mqo", skl.LinearRegression())])
    r2_certo.append(skm.cross_val_score(pipe, Xv, yv, cv=dobras, scoring="r2").mean())

print(f"R^2 verdadeiro           :  0.000")
print(f"PCA fora da dobra (errado): {np.mean(r2_errado):+.3f}")
print(f"PCA dentro do pipeline    : {np.mean(r2_certo):+.3f}")

> **A lição, e ela contraria o que se costuma dizer.** O PCA fora da dobra **não
> inflou** o $R^2$ — ele o **piorou**. Repetimos o experimento em quatro
> configurações de $(n, d, k)$ e a direção foi sempre a mesma.
>
> E o motivo é estrutural: **o PCA não olha o $y$.** A seleção de variáveis da Aula
> 07 fabricava $R^2 = +0{,}40$ porque escolhia as colunas que, por acaso, se
> pareciam com a resposta *naquela amostra* — é preciso ver o $y$ para conseguir
> isso. Sem $y$, não há como escolher direções que finjam explicar ruído.
>
> Quanto ao motivo de ficar *pior*: os componentes calculados sobre o conjunto todo
> não são os componentes ótimos de nenhuma dobra de treino, e o modelo ajustado
> sobre eles transfere pior. Fazer certo, aqui, além de honesto, dá um resultado
> melhor.

A conclusão prática não muda — ponha no `Pipeline`, custa uma linha — mas a
**classificação** muda: pré-processamento não supervisionado (padronização,
imputação pela média, PCA) pertence à categoria *leve* da hierarquia da Aula 07. O
que é grave é qualquer etapa que **olhe a resposta**.

---
## 8. PCA como pré-processamento: vale a pena?

A aplicação mais comum do PCA em aprendizado supervisionado é reduzir as colunas
antes de ajustar. Isso é uma aposta: você joga fora as direções de pouca variância
esperando que elas não sejam as que preveem $y$. **Não há garantia nenhuma disso** —
variância e utilidade preditiva são coisas diferentes.

Vamos medir, com uma busca que trata o número de componentes como hiperparâmetro.

In [ ]:
from sklearn.linear_model import LogisticRegression

X_tr, X_te, y_tr, y_te = skm.train_test_split(Xd, yd, test_size=0.3,
                                              random_state=0, stratify=yd)

sem_pca = Pipeline([("sc", StandardScaler()),
                    ("lg", LogisticRegression(max_iter=5000))]).fit(X_tr, y_tr)

com_pca = skm.GridSearchCV(
    Pipeline([("sc", StandardScaler()), ("pca", PCA()),
              ("lg", LogisticRegression(max_iter=5000))]),
    {"pca__n_components": [2, 5, 10, 20, 30, 40, 64]},
    cv=5, scoring="accuracy", n_jobs=-1).fit(X_tr, y_tr)

print(f"sem PCA (64 colunas)     : acuracia no teste = {sem_pca.score(X_te, y_te):.4f}")
print(f"com PCA, k escolhido = {com_pca.best_params_['pca__n_components']:2d}"
      f": acuracia no teste = {com_pca.score(X_te, y_te):.4f}")

res = pd.DataFrame({"componentes": com_pca.cv_results_["param_pca__n_components"],
                    "acuracia (CV)": com_pca.cv_results_["mean_test_score"]})
res.set_index("componentes").round(4)

A tabela responde com números, e a resposta aqui é desconfortável para quem espera
que o PCA ajude: a acurácia **cresce monotonicamente** com o número de componentes,
e a validação cruzada acaba escolhendo manter as 64 colunas. Reduzir dimensão custou
desempenho.

Não é um acidente, e a Seção 5 já tinha avisado: o PCA ordena direções por
**variância**, e variância não é o mesmo que utilidade preditiva. As direções de
pouca variância que ele descarta podem ser exatamente as que separam um 3 de um 8.

O que a tabela também mostra é que existe um bom **trade-off**: com 30 componentes
— menos da metade das colunas — perde-se cerca de um ponto de acurácia. Se o custo
computacional importa, essa é uma troca razoável. Se não importa, não há razão para
fazê-la.

Onde o PCA **realmente** ajuda em supervisionado é quando $d > n$, ou quando as
colunas são tão colineares que o modelo linear fica instável — que é o problema da
Aula 02, resolvido por outro caminho.

> **Sua vez.** Repita a comparação com um `KNeighborsClassifier` em vez da
> logística. Pense antes de rodar: pela Aula 05, o KNN sofre com dimensão alta e se
> beneficia de trabalhar na dimensão intrínseca. O PCA ajuda mais o KNN ou a
> logística?

---
## Resumo

| Conceito | Onde apareceu | O que vimos |
|---|---|---|
| eixos principais | §2 | direções ortogonais de máxima variância, com comprimento $\propto$ desvio |
| PCA $=$ SVD | §3 | as duas rotas dão o mesmo, a menos do **sinal**, que é arbitrário |
| variância explicada | §4 | `n_components=0.90` escolhe o número sozinho |
| reconstrução | §5 | 15 de 64 componentes já deixam os dígitos legíveis |
| componentes são imagens | §5 | cada CP é um padrão de traço, não uma coluna original |
| t-SNE | §6 | preserva 64% dos vizinhos (o PCA, 23%) e perde a estrutura global |
| não meça no t-SNE | §6 | é uma figura, não um espaço: nada de $k$-médias ou silhueta nas coordenadas dele |
| `perplexity` | §6 | muda o desenho e não tem validação cruzada que a escolha |
| PCA no pipeline | §7 | **não infla** o $R^2$ (ele não olha o $y$) — é vazamento leve, não grave |
| PCA antes de prever | §8 | nos dígitos, **custa** acurácia: variância $\neq$ utilidade preditiva |

**Leitura recomendada.** [AME] Capítulo 10 (redução de dimensionalidade). [ISLP]
§12.2 (componentes principais — §12.2.1 traz a interpretação como direção de máxima
variância e §12.2.3, a proporção de variância explicada) e §12.2.4, sobre quantos
componentes usar, que é honesto ao dizer que não há resposta objetiva. Para o
t-SNE, vale o artigo *"How to Use t-SNE Effectively"* (Wattenberg, Viégas &
Johnson, Distill 2016), que é interativo e mostra exatamente as armadilhas da
Seção 6.

**Para praticar.** `Lista teorica E2.pdf` (teórica, com gabarito) e
`Lista prática E2.ipynb` (prática, para completar as lacunas), nesta mesma
pasta.

**A seguir.** A Aula E3 fecha o curso aplicando tudo a texto — o domínio em que a
matriz de dados tem dezenas de milhares de colunas, quase todas zero, e em que a
maldição da dimensionalidade da Aula 05 aparece na forma mais pura.